In [1]:
%pip install -q gensim


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns

import gensim.downloader as api

from scipy.stats import pearsonr, spearmanr

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

np.random.seed(42)

In [3]:
model = api.load("glove-wiki-gigaword-300")
print(f"Словарь: {len(model.index_to_key)} слов, размерность: {model.vector_size}")

[==================================================] 100.0% 376.1/376.1MB downloaded
Словарь: 400000 слов, размерность: 300


In [22]:
## 1

import random

all_words = []

ks = random.sample(range(model.vector_size), 4)

for i in ks:
  i_vecs = []
  coord_values = model.vectors[:, i]
  top_idx = np.argsort(coord_values)[-10:][::-1]
  words = [model.index_to_key[j] for j in top_idx]
  all_words.append(words)

for word in all_words:
    print(*word)

medicare oblast savinja .833 uyezds benefits pensions inheritance inherited uyezd
madlyn 20-64 aonb jamborees henrieta galdeano euna kumgang yuanta 15-time
maale olshevsky herein qizheng poklonnaya muntz delavan arcore .769 bc-220
nodx palaeographically cepd 59-7 .313 kosdaq tecdax mérite eurodollar .455


В рамках одной координаты в эмббедингах смысловой связанности нет

In [63]:
## 2

d_vec = model["huge"] - model["small"]
d_norm = np.linalg.norm(d_vec)
d_normalized = d_vec / d_norm
x_d_vecs = model.vectors @ d_normalized
top_idx = np.argsort(x_d_vecs)
l_idx = top_idx[:10]
r_idx = top_idx[-10:]
l_words = [model.index_to_key[_] for _ in l_idx]
r_words = [model.index_to_key[_] for _ in r_idx]
print(*l_words)
print(*r_words)

nucleolar insectivorous small craterlets plumaged administaff passerine gastropod air-breathing medium-large
immense tremendous kovykta 787-billion 700-billion interés enormous shtokman massive huge


In [76]:
## 3

def cosine_similarity(x):
    return (model.vectors @ x) / (np.linalg.norm(model.vectors, axis=1) * np.linalg.norm(x))

v = model["cat"]
s = [
    model.similar_by_vector(v),
    model.similar_by_vector(v * 3),
    model.similar_by_vector(v * 0.1),
    model.similar_by_vector(v * -1)
]

for i in s:
    names = [i[0] for i in i]
    print(*names)

cat dog cats pet dogs feline monkey horse pets rabbit
cat dog cats pet dogs feline monkey horse pets rabbit
cat dog cats pet dogs feline monkey horse pets rabbit
nucleation 2,780 i-29 pichler 27,100 gordeev ananiev nozzolio tollner overinflated


In [104]:
drinks = ["tea", "coffee", "juice", "beer", "wine"]

average_vecs = np.average(model[drinks], axis=0)
s = model.similar_by_vector(average_vecs, topn=15)

n = [name for name, _ in s if name not in drinks]
print(*n)

drink drinks fruit beverage wines bottled milk bottle sipping vodka


In [105]:
random_words = ["code", "senator", "pen", "volcano", "cat"]

average_vecs = np.average(model[random_words], axis=0)
s = model.similar_by_vector(average_vecs, topn=15)

n = [name for name, _ in s if name not in random_words]
print(*n)

dog name known called example instance same like word using even


In [119]:
drinks_centroid = np.average(model[drinks], axis=0)
drinks_norm_of_centroid = np.linalg.norm(drinks_centroid)
print(f"Норма центроида (drinks): {drinks_norm_of_centroid}")

drinks_norm = np.linalg.norm(model[drinks], axis=1)
avg_drinks_norm = np.mean(drinks_norm)
print(f"Норма набора (drinks): {avg_drinks_norm}")
print(avg_drinks_norm / drinks_norm_of_centroid)

print()

random_centroid = np.average(model[random_words], axis=0)
random_norm_of_centroid = np.linalg.norm(random_centroid)
print(f"Норма центроида (random): {random_norm_of_centroid}")

random_norm = np.linalg.norm(model[random_words], axis=1)
avg_random_norm = np.mean(random_norm)
print(f"Норма набора (random): {avg_random_norm}")
print(avg_random_norm / random_norm_of_centroid)

Норма центроида (drinks): 5.501314640045166
Норма набора (drinks): 7.1434807777404785
1.2985044

Норма центроида (random): 3.4068925380706787
Норма набора (random): 6.798051357269287
1.9953818


In [ ]:
def analogy(a, b, c):
